# 02 · Train and evaluate everything (results saved to Drive)
Re-run safely after a disconnect: deep models resume from their last checkpoint in Drive, IMS features are
cached, and every report is written to `experiments/comparison_reports/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Installs the project code into Drive the first time (upload dist/qadt_code.zip built with
# scripts/make_colab_bundle.sh), then reuses it in every later session.
import os, sys, zipfile
CODE_DIR = '/content/drive/MyDrive/Quantum_Agentic_Digital_Twin_code'
if not os.path.isdir(os.path.join(CODE_DIR, 'backend')):
    print(f'Project code not found in {CODE_DIR}. Choose qadt_code.zip to upload it:')
    from google.colab import files
    uploaded = files.upload()
    if not uploaded:
        raise SystemExit('No file uploaded. Build it locally with: scripts/make_colab_bundle.sh')
    os.makedirs(CODE_DIR, exist_ok=True)
    with zipfile.ZipFile(next(iter(uploaded))) as z:
        z.extractall(CODE_DIR)
    print('Code installed in Drive at', CODE_DIR)
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
%pip install -q -r requirements.txt
!apt-get -qq install -y unrar > /dev/null   # the IMS archive contains .rar files
print('Ready:', CODE_DIR)

In [ ]:
from backend.storage import setup_storage, prepare_datasets, require_dataset
setup_storage(); prepare_datasets()
require_dataset('cmapss'); require_dataset('ims_bearings')

## C-MAPSS RUL models (8 models; deep models ≈ 1–2 min each on CPU)

In [ ]:
!python scripts/train_all_rul.py FD001 --epochs 40

## IMS anomaly detectors (feature extraction over ~9,500 snapshots, then 4 detectors × 3 tests + cross-test supervised)

In [ ]:
!python scripts/train_all_anomaly.py

## Quantum experiments

In [ ]:
from backend.quantum_models.qml import QuantumConfig, run_experiment
for model in ['quantum_kernel', 'vqc', 'qnn_regressor']:
    for q in (4, 6):
        r = run_experiment(QuantumConfig(model=model, n_qubits=q, layers=2, n_train=200, epochs=20))
        c = r['comparison']
        print(f"{model:15} {q}q  {c['metric']} quantum={c['quantum']:.3f}  best classical {c['best_classical_model']}={c['best_classical']:.3f}  depth={r['quantum']['circuit_depth']}")

## Explanations and a QAOA scheduling run

In [ ]:
from backend.explainability.explain import explain_rul_model
from backend.services import fleet
from backend.langchain_tools.tools import optimize_maintenance_schedule
from backend.storage import save_result
for m in ['xgboost', 'catboost', 'lstm']:
    explain_rul_model(fleet.rul_model(m, 'FD001'), fleet.cmapss_data('FD001'))
res = optimize_maintenance_schedule(fleet.default_assets('FD001', 5))
save_result('qaoa_results', 'schedule_notebook', res)
for r in res['results']:
    print(f"{r['solver']:20} cost={r.get('total_expected_cost', 0):>12,.0f}  gap={r.get('gap_percent', 0):6.2f}%")

## Storage page

In [ ]:
from backend.storage import show_storage_page
show_storage_page()